# Stage 2b: BanglaT5 Encoder-Decoder Full Fine-Tuning

This notebook implements a full fine-tune of **`csebuetnlp/banglat5`** (247M parameters, T5-base encoder-decoder architecture) on `sft_train.csv`.

### Key Architectural Rationale
- **Encoder-Decoder Superiority on Constrained NLG**: Text generation scored by BERTScore, Token F1, and ROUGE-L is structurally a translation/summarization benchmark. Encoder-decoder models excel at grounded sequence-to-sequence generation.
- **BanglaT5 Benchmarks**: Published research demonstrates BanglaT5 outperforms mT5-base by up to +9.5% on QA-style Bangla generation despite having fewer than half the parameters.
- **Full Fine-Tuning Efficiency**: At 247M parameters, BanglaT5 is fast to train end-to-end (no LoRA adapter needed) and falls safely within the competition's 3B parameter ceiling.

### 1. Environment Setup & Bengali Normalizer Verification
Install `csebuetnlp/normalizer`, `transformers`, `datasets`, `accelerate`, and `sentencepiece`. We immediately test the normalizer on 3 diverse Bengali sentences from the dataset.

In [ ]:
!pip install -q git+https://github.com/csebuetnlp/normalizer transformers datasets accelerate sentencepiece matplotlib pandas

import os
import sys
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt

try:
    from normalizer import normalize
    print("SUCCESS: csebuetnlp/normalizer loaded successfully.")
except ImportError:
    print("WARNING: normalizer package not found, using identity fallback.")
    def normalize(text):
        return str(text)

# Test normalizer on 3 hand-picked Bengali sentences with numbers, punctuation, and medical terms
sample_sentences = [
    "আমার বাচ্চার ১০২° জ্বর এবং কাশি হচ্ছে, সাথে প্যারাসিটামল ৫০০mg দেওয়া যাবে কি?",
    "খাবারের পর পর পেটে গ্যাস & বুক জ্বালাপোড়া করে... কী ঔষধ খাব?!",
    "ডায়াবেটিস লেভেল ১২.৫ mmol/L দেখাচ্ছে; ইনসুলিনের মাত্রা কত হওয়া উচিত??"
]

print("\n=== Normalizer Verification Test (Before vs After) ===")
for i, sent in enumerate(sample_sentences, 1):
    norm_sent = normalize(sent)
    print(f"\nSample {i}:")
    print(f"  Original:   {sent}")
    print(f"  Normalized: {norm_sent}")

### 2. Load BanglaT5 Model & Tokenizer and Verify Parameter Cap
Load `csebuetnlp/banglat5` and verify that the parameter count is in the ~200-300M range, well below the 3 Billion parameter limit.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "csebuetnlp/banglat5"
print(f"Loading model and tokenizer: {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

param_count = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n=== Parameter Count Verification ===")
print(f"Total Parameters:     {param_count:,}")
print(f"Trainable Parameters: {trainable_params:,}")
if param_count <= 3_000_000_000:
    print(f"RESULT: {param_count:,} parameters is well under the 3B cap: PASS")
else:
    raise ValueError(f"Model exceeds 3B limit: {param_count:,}")

### 3. GPU Detection & Hardware Configuration

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
major_cc = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
use_bf16 = torch.cuda.is_available() and major_cc >= 8 and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

# Default batch size 16 for T5-base
per_device_batch_size = 16
gradient_accumulation_steps = 1

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Detected GPU: {gpu_name} ({gpu_vram_gb:.2f} GB VRAM, Compute Capability {major_cc}.{torch.cuda.get_device_capability()[1]})")
    if gpu_vram_gb < 14.0:
        per_device_batch_size = 8
        gradient_accumulation_steps = 2
        print("Adjusted batch size to 8 (accum steps 2) for lower VRAM.")
else:
    print("Running on CPU (not recommended for full training).")

print(f"Configuration: Batch Size = {per_device_batch_size}, Grad Accum = {gradient_accumulation_steps}, bf16 = {use_bf16}, fp16 = {use_fp16}")

### 4. Data Preprocessing & Verification
- Load `sft_train.csv` (columns: `id`, `input`, `output`)
- Apply `normalize()` to both input and output text
- Tokenize inputs with `max_length=384`
- Tokenize outputs with `max_length=300`, setting padding token labels to `-100` so they are ignored in cross-entropy loss calculation
- Verify 2 decoded samples

In [ ]:
from datasets import Dataset
import datasets
datasets.disable_caching()
import datasets.arrow_dataset
datasets.arrow_dataset.generate_fingerprint = lambda *args, **kwargs: 'mock_fingerprint'

train_path = "working/sft_train.csv" if os.path.exists("working/sft_train.csv") else "sft_train.csv"
val_path = "working/sft_val.csv" if os.path.exists("working/sft_val.csv") else "sft_val.csv"

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

print(f"Loaded train set: {len(df_train)} rows | val set: {len(df_val)} rows")

# Sample 200 rows from sft_val.csv for fast epoch-by-epoch evaluation during training
eval_sample_df = df_val.sample(n=min(200, len(df_val)), random_state=42).reset_index(drop=True)
print(f"Created fast eval subset: {len(eval_sample_df)} rows")

def preprocess_function(examples):
    inputs = [normalize(str(x).strip()) for x in examples["input"]]
    targets = [normalize(str(y).strip()) for y in examples["output"]]
    
    # Tokenize inputs
    model_inputs = tokenizer(inputs, max_length=384, truncation=True)
    
    # Tokenize labels
    labels = tokenizer(text_target=targets, max_length=300, truncation=True)
    
    # Replace pad token id with -100 to ignore pad tokens in loss
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

raw_train_ds = Dataset.from_pandas(df_train[["input", "output"]])
raw_eval_ds = Dataset.from_pandas(eval_sample_df[["input", "output"]])

train_tokenized = raw_train_ds.map(preprocess_function, batched=True, remove_columns=raw_train_ds.column_names)
eval_tokenized = raw_eval_ds.map(preprocess_function, batched=True, remove_columns=raw_eval_ds.column_names)

# Print 2 fully decoded examples to verify tokenization correctness
print("\n=== Verification of Tokenized Examples ===")
for i in range(2):
    sample = train_tokenized[i]
    decoded_input = tokenizer.decode(sample["input_ids"], skip_special_tokens=False)
    label_ids = [l if l != -100 else tokenizer.pad_token_id for l in sample["labels"]]
    decoded_label = tokenizer.decode(label_ids, skip_special_tokens=True)
    print(f"\n--- Example {i+1} ---")
    print(f"Decoded Input:  {decoded_input[:120]}...")
    print(f"Decoded Target: {decoded_label[:120]}...")

### 5. Training Setup (Seq2SeqTrainer)
Hyperparameters mirror the official BanglaT5 paper's fine-tuning recipe for Bengali text generation:
- `num_train_epochs = 6`
- `learning_rate = 3e-4`
- `warmup_ratio = 0.1`
- `label_smoothing_factor = 0.1`
- `weight_decay = 1e-6`
- `load_best_model_at_end = True` (metric: `eval_loss`)
- Dynamic batch collator (`DataCollatorForSeq2Seq`)

In [ ]:
import inspect
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

output_dir = "working/banglat5_ckpt"
os.makedirs(output_dir, exist_ok=True)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

def build_training_args(bs, grad_acc):
    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        num_train_epochs=6,
        learning_rate=3e-4,
        warmup_ratio=0.1,
        label_smoothing_factor=0.1,
        weight_decay=1e-6,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs,
        gradient_accumulation_steps=grad_acc,
        predict_with_generate=True,
        generation_max_length=300,
        bf16=use_bf16,
        fp16=use_fp16,
        logging_steps=100,
        report_to="none"
    )

def build_seq2seq_trainer(model, args, train_dataset, eval_dataset, tokenizer, data_collator):
    kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_dataset,
        "eval_dataset": eval_dataset,
        "data_collator": data_collator,
    }
    sig = inspect.signature(Seq2SeqTrainer.__init__).parameters
    if "processing_class" in sig:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sig:
        kwargs["tokenizer"] = tokenizer
    return Seq2SeqTrainer(**kwargs)

training_args = build_training_args(per_device_batch_size, gradient_accumulation_steps)

trainer = build_seq2seq_trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("Trainer initialized successfully.")

### 6. Execute Fine-Tuning with Automatic OOM Handling

In [ ]:
start_train_time = time.time()

try:
    print("Starting BanglaT5 fine-tuning...")
    train_result = trainer.train()
except torch.cuda.OutOfMemoryError as e:
    print("\n[WARNING] CUDA OOM encountered! Halving batch size and doubling gradient accumulation steps...")
    torch.cuda.empty_cache()
    per_device_batch_size = max(1, per_device_batch_size // 2)
    gradient_accumulation_steps *= 2
    print(f"Retrying with Batch Size: {per_device_batch_size}, Grad Accum: {gradient_accumulation_steps}")
    training_args = build_training_args(per_device_batch_size, gradient_accumulation_steps)
    trainer = build_seq2seq_trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=eval_tokenized,
        tokenizer=tokenizer,
        data_collator=data_collator
    )
    train_result = trainer.train()

total_train_time = time.time() - start_train_time
print(f"\nTraining complete in {total_train_time/60:.2f} minutes.")

### 7. Plot Loss Curves

In [ ]:
log_history = trainer.state.log_history

train_losses = [x["loss"] for x in log_history if "loss" in x]
eval_losses = [x["eval_loss"] for x in log_history if "eval_loss" in x]

plt.figure(figsize=(9, 5))
if train_losses:
    plt.plot(range(1, len(train_losses) + 1), train_losses, label="Train Loss", color="#1f77b4", lw=2)
if eval_losses:
    eval_steps = [x["step"] for x in log_history if "eval_loss" in x]
    plt.plot(range(1, len(eval_losses) + 1), eval_losses, label="Validation Loss (Per Epoch)", color="#ff7f0e", marker="o", lw=2)

plt.title("BanglaT5 Fine-Tuning Loss Curve", fontsize=14, fontweight="bold")
plt.xlabel("Epoch / Log Step", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### 8. Inspect 5 Held-Out Sample Generations (Side-by-Side with Ground Truth)
Using BanglaT5 paper's beam search decoding recipe (`num_beams=5`, `no_repeat_ngram_size=3`, `length_penalty=0.6`, `max_new_tokens=300`).

In [ ]:
model.eval()
model.to(device)

print("=== BanglaT5 Generation Quality Check on 5 Held-Out Validation Samples ===\n")
sample_eval_rows = df_val.head(5)

for idx, row in sample_eval_rows.iterrows():
    p_input = normalize(str(row["input"]).strip())
    p_ref = str(row["output"]).strip()
    
    inputs = tokenizer(p_input, return_tensors="pt", max_length=384, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=5,
            no_repeat_ngram_size=3,
            length_penalty=0.6,
            max_new_tokens=300,
            min_new_tokens=40,
            early_stopping=True
        )
        
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    print(f"------------------ Sample {idx + 1} (ID: {row['id']}) ------------------")
    print(f"INPUT:     {p_input}")
    print(f"REFERENCE: {p_ref}")
    print(f"PREDICTED: {pred_text}\n")

### 9. Save Final Model & Tokenizer Checkpoint

In [ ]:
final_save_dir = "working/banglat5_final"
os.makedirs(final_save_dir, exist_ok=True)

print(f"Saving best BanglaT5 checkpoint to {final_save_dir}...")
model.save_pretrained(final_save_dir)
tokenizer.save_pretrained(final_save_dir)
print("Model and tokenizer saved successfully.")

### 10. Training Diagnostics & Summary

In [ ]:
def get_dir_size_mb(path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)

model_size_mb = get_dir_size_mb(final_save_dir)
final_train_loss = train_losses[-1] if train_losses else "N/A"
final_eval_loss = eval_losses[-1] if eval_losses else "N/A"

print("==========================================================")
print("         BANGLAT5 FULL FINE-TUNE SUMMARY")
print("==========================================================")
print(f"Saved Model Directory:   {final_save_dir}")
print(f"Disk Usage:              {model_size_mb:.2f} MB")
print(f"Final Training Loss:     {final_train_loss}")
print(f"Final Validation Loss:   {final_eval_loss}")
print(f"Total Training Time:     {int(total_train_time // 60)}m {int(total_train_time % 60)}s")
print(f"Parameter Count:         {param_count:,} (PASS, < 3B)")
print("==========================================================")